# Correcting tune 

##  Import of modules

In [23]:
import logging

In [ ]:
Activate the warning level below, otherwise you will see a lot of warnings

In [ ]:
# logging.basicConfig(level=logging.warning)

In [1]:
import jsons
import yaml

In [2]:
from accml_lib.core.bl.delta_backend import StateCache
from accml_lib.core.bl.command_rewritter import CommandRewriter
from accml_lib.core.model.output.tune import Tune
from accml_lib.core.model.utils.command import ReadCommand

In [3]:
from accml_lib.custom.bessyii.liasion_translator_setup import load_managers
from accml_lib.custom.bessyii.setup import setup

/home/mfp/workspace/accml-main-lib-splitup/accml/venv-accml-tst/lib/python3.10/site-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
  warn(


Extend DSO search path to '/home/mfp/workspace/accml-main-lib-splitup/accml/venv-accml-tst/lib/python3.10/site-packages/epicscorelibs/lib'
Extend DSO search path to '/home/mfp/workspace/accml-main-lib-splitup/accml/venv-accml-tst/lib/python3.10/site-packages/pvxslibs/lib'


In [4]:
from accml.custom.ophyd_async.ophyd_async_backend import OphydAsyncDeviceBackendRW
from accml.custom.ophyd_async.ophyd_async_delta_backend import OphydAsyncDeltaBackendRWProxy

In [5]:
from accml.core.utils.simple_storage import SimpleDataStorage
from accml.core.utils.basic_measurement_execution_engine import BasicMeasurementExecutionEngine

In [6]:
from accml.app.tune.model import TuneResponseCollection
from accml.app.tune.oracle import TuneOracle
from accml.app.tune.policy import TunePolicy
from accml.app.tune.tune_correction_controller import TuneCorrectionController

In [7]:
import asyncio

## Setup of imputs

In [8]:
!find  ../../ -name '*.yml';
!pwd

../../notebooks/tune/tune_response_data_from_simulator.yml
../../03_reference_data/tune_response_from_simulation.yml
/home/mfp/workspace/accml-main-lib-splitup/accml/examples/notebooks/tune


In [11]:
with open("../../03_reference_data/tune_response_from_simulation.yml") as fp:
    d = yaml.load(fp, yaml.SafeLoader)
dm = jsons.load(d, TuneResponseCollection)

In [12]:
yp, lm, ts = load_managers()
devices = setup(prefix=None)

Loading data path custom/config_data/bessyii/magnets.yaml from /home/mfp/workspace/accml-main-lib-splitup/accml/external-repositories/accml_lib/src/accml_lib/custom/config_data/bessyii/magnets.yaml
Loading data path custom/config_data/bessyii/power_converters.yaml from /home/mfp/workspace/accml-main-lib-splitup/accml/external-repositories/accml_lib/src/accml_lib/custom/config_data/bessyii/power_converters.yaml
using prefix=mfp:


In [13]:
backend = OphydAsyncDeltaBackendRWProxy(
    OphydAsyncDeviceBackendRW(devices=devices),
    cache=StateCache(name="BESSY2_OphydAsync_Dev_State_Cache"),
)

In [14]:
mexec = BasicMeasurementExecutionEngine(
    backend=backend,
    cmd_rewriter=CommandRewriter(liaison_manager=lm, translation_service=ts),
    storage=SimpleDataStorage(),
    expected_view_for_output="device",
    num_readings=2,
)

In [15]:
tune_target = Tune(x=1065, y=855)

In [16]:
oracle = TuneOracle(col=dm, target=tune_target)

In [17]:
policy = TunePolicy(scale=1.0)

In [18]:
controller = TuneCorrectionController(
    oracle=oracle,
    policy=policy,
    mexec=mexec,
    wait_after_set=1.2,
    wait_between_samples=0.5,
    n_samples=3
)

In [19]:
rcmds = [ReadCommand(id="tune", property="transversal")]
set_cmds = [
        ReadCommand(id=elm.pc_name, property="delta_set_current") for elm in dm.col
]

In [20]:
await asyncio.gather(*[devices.get(rc.id).connect() for rc in rcmds])

Connecting to ca://mfp:TUNEZR:rdH
Connecting to ca://mfp:TUNEZR:rdV


[None]

In [21]:
await asyncio.gather(*[devices.get(rc.id).connect() for rc in set_cmds]);

Connecting to ca://mfp:Q3PD2R:set
Connecting to ca://mfp:Q3PD2R:rdbk
Connecting to ca://mfp:Q3PD2R:rdbk.EGU
Connecting to ca://mfp:Q3PD2R:rdbk.PREC
Connecting to ca://mfp:Q4P1T1R:set
Connecting to ca://mfp:Q4P1T1R:rdbk
Connecting to ca://mfp:Q4P1T1R:rdbk.EGU
Connecting to ca://mfp:Q4P1T1R:rdbk.PREC
Connecting to ca://mfp:Q4PD1R:set
Connecting to ca://mfp:Q4PD1R:rdbk
Connecting to ca://mfp:Q4PD1R:rdbk.EGU
Connecting to ca://mfp:Q4PD1R:rdbk.PREC
Connecting to ca://mfp:Q3PT4R:set
Connecting to ca://mfp:Q3PT4R:rdbk
Connecting to ca://mfp:Q3PT4R:rdbk.EGU
Connecting to ca://mfp:Q3PT4R:rdbk.PREC
Connecting to ca://mfp:Q3PD6R:set
Connecting to ca://mfp:Q3PD6R:rdbk
Connecting to ca://mfp:Q3PD6R:rdbk.EGU
Connecting to ca://mfp:Q3PD6R:rdbk.PREC
Connecting to ca://mfp:Q4PD2R:set
Connecting to ca://mfp:Q4PD2R:rdbk
Connecting to ca://mfp:Q4PD2R:rdbk.EGU
Connecting to ca://mfp:Q4PD2R:rdbk.PREC
Connecting to ca://mfp:Q3PD3R:set
Connecting to ca://mfp:Q3PD3R:rdbk
Connecting to ca://mfp:Q3PD3R:rdbk.EGU


In [22]:
await controller.continuous(read_commands=rcmds, set_commands=set_cmds, n_steps=2)

Making subscription on source ca://mfp:TUNEZR:rdV
Making subscription on source ca://mfp:TUNEZR:rdH
Updated subscription: reading of source ca://mfp:TUNEZR:rdV changed from None to {'value': 905.1455276751433, 'timestamp': 1769859207.432757, 'alarm_severity': 0}
Updated subscription: reading of source ca://mfp:TUNEZR:rdH changed from None to {'value': 1070.4086221493847, 'timestamp': 1769859207.419433, 'alarm_severity': 0}
Updated subscription: reading of source ca://mfp:TUNEZR:rdH changed from {'value': 1070.4086221493847, 'timestamp': 1769859207.419433, 'alarm_severity': 0} to {'value': 1070.408622148984, 'timestamp': 1769859208.459945, 'alarm_severity': 0}
Updated subscription: reading of source ca://mfp:TUNEZR:rdV changed from {'value': 905.1455276751433, 'timestamp': 1769859207.432757, 'alarm_severity': 0} to {'value': 905.1455276760832, 'timestamp': 1769859208.461635, 'alarm_severity': 0}
Closing subscription on source ca://mfp:TUNEZR:rdH
Closing subscription on source ca://mfp:T

Read tune (from mexec) Tune(x=1070.4086, y=905.1455), diff Tune(x=-5.4086, y=-50.1455)
Oracle correction action -0.1553 +/-  0.1528 range: -0.3665 --  0.1002
Applying          action -0.1553 +/-  0.1528 range: -0.3665 --  0.1002


Putting value 190.74257506855682 to backend at source ca://mfp:Q3PD2R:set
Putting value 251.99061497605757 to backend at source ca://mfp:Q4P1T1R:set
Putting value 131.11622326329626 to backend at source ca://mfp:Q4PD1R:set
Putting value 221.44952164223383 to backend at source ca://mfp:Q3PT4R:set
Putting value 188.87221398003916 to backend at source ca://mfp:Q3PD6R:set
Putting value 138.27065367138846 to backend at source ca://mfp:Q4PD2R:set
Putting value 190.33662570787155 to backend at source ca://mfp:Q3PD3R:set
Putting value 223.68002013615083 to backend at source ca://mfp:Q3P2T1R:set
Putting value 213.54228626464558 to backend at source ca://mfp:Q3P2T6R:set
Putting value 245.14749220869675 to backend at source ca://mfp:Q4P1T8R:set
Putting value 245.22626554892426 to backend at source ca://mfp:Q4P2T6R:set
Putting value 245.3714490226544 to backend at source ca://mfp:Q4P2T1R:set
Putting value 217.2373130548991 to backend at source ca://mfp:Q4P1T6R:set
Putting value 246.78284616804126 

Read tune (from mexec) Tune(x=1065.1458, y=855.1785), diff Tune(x=-0.1458, y=-0.1785)
Oracle correction action -0.1562 +/-  0.1534 range: -0.3682 --  0.1003
Applying          action -0.1562 +/-  0.1534 range: -0.3682 --  0.1003


Putting value 190.7409080445885 to backend at source ca://mfp:Q3PD2R:set
Putting value 251.9901525869763 to backend at source ca://mfp:Q4P1T1R:set
Putting value 131.11556781245693 to backend at source ca://mfp:Q4PD1R:set
Putting value 221.44833033793373 to backend at source ca://mfp:Q3PT4R:set
Putting value 188.87062429388743 to backend at source ca://mfp:Q3PD6R:set
Putting value 138.27054698463806 to backend at source ca://mfp:Q4PD2R:set
Putting value 190.3349318447804 to backend at source ca://mfp:Q3PD3R:set
Putting value 223.6793506134517 to backend at source ca://mfp:Q3P2T1R:set
Putting value 213.54164653632276 to backend at source ca://mfp:Q3P2T6R:set
Putting value 245.14678174754192 to backend at source ca://mfp:Q4P1T8R:set
Putting value 245.2258839846831 to backend at source ca://mfp:Q4P2T6R:set
Putting value 245.37075973845322 to backend at source ca://mfp:Q4P2T1R:set
Putting value 217.23646857568667 to backend at source ca://mfp:Q4P1T6R:set
Putting value 246.78111099556975 to 